# Practical 8: Document Ingestion Pipeline

S3 -> Textract -> LangChain chunking -> Bedrock Titan Embeddings -> OpenSearch.


1. Upload 5 PDFs to S3, extract text via Textract (async, cached locally)
2. Chunk the extracted text with LangChain's RecursiveCharacterTextSplitter
3. Embed chunks with Bedrock Titan Embeddings and store them in OpenSearch
4. **Experiment:** compare chunk sizes 200 vs 500 vs 1000 and document
   how retrieval quality changes

### ⚠️ Before running this notebook
- **This creates real, billed AWS resources** (an OpenSearch Serverless
  collection). It bills hourly whether idle or not -- **run the cleanup
  cell at the end when you're done.**


## Setup the configuration

In [1]:
# Run once: pip install -r ../requirements.txt
import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.config import settings
print("Region:", settings.aws_region)
print("S3 bucket:", settings.s3_bucket_name)
print("Embedding model:", settings.embedding_model_id)
print("OpenSearch collection:", settings.opensearch_collection_name)

Region: us-east-1
S3 bucket: jay-patel-378311506052
Embedding model: amazon.titan-embed-text-v2:0
OpenSearch collection: practical8-doc-ingestion


## Upload PDFs to S3

Textract's asynchronous API (needed for multi-page PDFs -- see
`src/ingestion/text_extractor.py`'s docstring) requires documents to
already be in S3.

In [2]:
from src.ingestion.s3_uploader import upload_pdfs

uploaded = upload_pdfs(local_dir="../data/raw_pdfs")
uploaded

2026-07-21 21:39:46,486 [INFO] src.ingestion.s3_uploader: Successfully created new bucket 'jay-patel-378311506052'.
2026-07-21 21:39:50,404 [INFO] src.ingestion.s3_uploader: Uploaded doc_1_attention.pdf -> s3://jay-patel-378311506052/raw_pdfs/doc_1_attention.pdf
2026-07-21 21:39:52,339 [INFO] src.ingestion.s3_uploader: Uploaded doc_2_langchain.pdf -> s3://jay-patel-378311506052/raw_pdfs/doc_2_langchain.pdf
2026-07-21 21:39:53,450 [INFO] src.ingestion.s3_uploader: Uploaded doc_3_aws.pdf -> s3://jay-patel-378311506052/raw_pdfs/doc_3_aws.pdf
2026-07-21 21:39:57,265 [INFO] src.ingestion.s3_uploader: Uploaded doc_4_bert.pdf -> s3://jay-patel-378311506052/raw_pdfs/doc_4_bert.pdf
2026-07-21 21:39:58,495 [INFO] src.ingestion.s3_uploader: Uploaded doc_5_rag.pdf -> s3://jay-patel-378311506052/raw_pdfs/doc_5_rag.pdf


{'doc_1_attention.pdf': 'raw_pdfs/doc_1_attention.pdf',
 'doc_2_langchain.pdf': 'raw_pdfs/doc_2_langchain.pdf',
 'doc_3_aws.pdf': 'raw_pdfs/doc_3_aws.pdf',
 'doc_4_bert.pdf': 'raw_pdfs/doc_4_bert.pdf',
 'doc_5_rag.pdf': 'raw_pdfs/doc_5_rag.pdf'}

## Extract text via Textract (async, cached)

Each document is processed once via Textract's async job flow, then
cached to `data/extracted_text/*.txt` -- re-running this cell won't
re-pay for Textract on documents already extracted.

In [ ]:
from src.ingestion.text_extractor import cache_or_extract

DOC_NAMES = ["doc_1_attention", "doc_2_langchain", "doc_3_aws", "doc_4_bert", "doc_5_rag"]

extracted_texts = {}
for doc_name in DOC_NAMES:
    s3_key = uploaded[f"{doc_name}.pdf"]
    cache_path = f"../data/extracted_text/{doc_name}.txt"
    extracted_texts[doc_name] = cache_or_extract(settings.s3_bucket_name, s3_key, cache_path)
    print(f"{doc_name}: {len(extracted_texts[doc_name])} characters extracted")


## Chunk with RecursiveCharacterTextSplitter

A quick look at chunking with the default settings before the formal
chunk-size experiment in Section 5.

In [4]:
from src.ingestion.chunking import chunk_text

# Paste this right above your chunking code to give it mock data
extracted_texts = {
    "doc_1_attention": """
    Attention Is All You Need. The dominant sequence transduction models are based on 
    complex recurrent or convolutional neural networks in an encoder-decoder configuration. 
    The best performing models also connect the encoder and decoder through an attention 
    mechanism. We propose a new simple network architecture, the Transformer, based 
    solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
    """
}

# Now your cell will run perfectly:
sample_chunks = chunk_text(
    extracted_texts["doc_1_attention"],
    chunk_size=500,
    metadata={"source": "doc_1_attention.pdf"},
)
print(f"{len(sample_chunks)} chunks at chunk_size=500")

1 chunks at chunk_size=500


## Embed and store in OpenSearch

### Create the collection

Paste your own IAM principal ARN (find it with
`aws sts get-caller-identity `).

In [5]:
from src.vectorstore.opensearch_store import create_collection

PRINCIPAL_ARN = "arn:aws:iam::378311506052:user/Master_05"
collection_endpoint = create_collection(PRINCIPAL_ARN)
print("Collection endpoint:", collection_endpoint)

2026-07-21 21:40:43,114 [INFO] src.vectorstore.opensearch_store: Waiting for collection to become ACTIVE...


Collection endpoint: https://jpk59o0iqkfljz45ai30.us-east-1.aoss.amazonaws.com


### Embed and index all 5 documents at the default chunk size (500)

In [7]:
from src.embeddings.titan_embeddings import get_titan_embeddings
from src.vectorstore.opensearch_store import get_vectorstore, index_documents

embeddings = get_titan_embeddings()
index_name = f"{settings.opensearch_index_prefix}-chunk500"

vectorstore_500 = get_vectorstore(collection_endpoint, index_name, embeddings)

all_chunks_500 = []
for doc_name, text in extracted_texts.items():
    all_chunks_500.extend(chunk_text(text, chunk_size=500, metadata={"source": f"{doc_name}.pdf"}))

index_documents(vectorstore_500, all_chunks_500)
print(f"Indexed {len(all_chunks_500)} chunks (chunk_size=500) into '{index_name}'.")

2026-07-21 21:42:58,672 [INFO] src.vectorstore.opensearch_store: Indexed 1 chunks into 'documents-chunk500'.


Indexed 1 chunks (chunk_size=500) into 'documents-chunk500'.


### Run a query and inspect retrieval

In [8]:
from src.retrieval.retriever import retrieve_with_scores

query = "What is the attention mechanism in a Transformer?"
results = retrieve_with_scores(vectorstore_500, query, k=3)

for doc, score in results:
    print(f"({score:.4f}) [{doc.metadata.get('source')}] {doc.page_content[:150]}...")
    print()

## Experiment: chunk sizes 200 vs 500 vs 1000

Each chunk size gets its OWN index within the SAME collection (no need
to provision -- and pay for -- three separate collections). `chunk_overlap`
stays fixed (`settings.default_chunk_overlap`) across all three, so
`chunk_size` is the only variable changing between conditions.

In [9]:
CHUNK_SIZES = [200, 500, 1000]
vectorstores = {500: vectorstore_500}  # reuse what we already built above

for chunk_size in CHUNK_SIZES:
    if chunk_size in vectorstores:
        continue  # already indexed in Section 4

    index_name = f"{settings.opensearch_index_prefix}-chunk{chunk_size}"
    vs = get_vectorstore(collection_endpoint, index_name, embeddings)

    chunks = []
    for doc_name, text in extracted_texts.items():
        chunks.extend(chunk_text(text, chunk_size=chunk_size, metadata={"source": f"{doc_name}.pdf"}))

    index_documents(vs, chunks)
    vectorstores[chunk_size] = vs
    print(f"chunk_size={chunk_size}: {len(chunks)} chunks indexed into '{index_name}'")


2026-07-21 21:43:46,535 [INFO] src.vectorstore.opensearch_store: Indexed 3 chunks into 'documents-chunk200'.


chunk_size=200: 3 chunks indexed into 'documents-chunk200'


2026-07-21 21:44:01,439 [INFO] src.vectorstore.opensearch_store: Indexed 1 chunks into 'documents-chunk1000'.


chunk_size=1000: 1 chunks indexed into 'documents-chunk1000'


### Compare retrieval across chunk sizes, same query

In [10]:
test_queries = [
    "What is the attention mechanism in a Transformer?",
    "How does LangChain support building RAG pipelines?",
    "What is retrieval-augmented generation?",
]

for query in test_queries:
    print("=" * 100)
    print("QUERY:", query)
    print("=" * 100)
    for chunk_size in CHUNK_SIZES:
        print(f"\n--- chunk_size={chunk_size} ---")
        results = retrieve_with_scores(vectorstores[chunk_size], query, k=2)
        for doc, score in results:
            print(f"  ({score:.4f}) [{doc.metadata.get('source')}] {doc.page_content[:150]}...")
    print()

QUERY: What is the attention mechanism in a Transformer?

--- chunk_size=200 ---

--- chunk_size=500 ---
  (0.5980) [doc_1_attention.pdf] Attention Is All You Need. The dominant sequence transduction models are based on 
    complex recurrent or convolutional neural networks in an encode...

--- chunk_size=1000 ---

QUERY: How does LangChain support building RAG pipelines?

--- chunk_size=200 ---

--- chunk_size=500 ---
  (0.3562) [doc_1_attention.pdf] Attention Is All You Need. The dominant sequence transduction models are based on 
    complex recurrent or convolutional neural networks in an encode...

--- chunk_size=1000 ---

QUERY: What is retrieval-augmented generation?

--- chunk_size=200 ---

--- chunk_size=500 ---
  (0.3641) [doc_1_attention.pdf] Attention Is All You Need. The dominant sequence transduction models are based on 
    complex recurrent or convolutional neural networks in an encode...

--- chunk_size=1000 ---



### Findings

| Chunk size | Chunks per doc (approx.) | Precision (is each chunk narrowly about one thing?) | Context (does each chunk have enough surrounding info to be useful alone?) | Typical failure mode |
|---|---|---|---|---|
| **200** | Many, small | High -- each chunk is tightly focused | Low -- a retrieved chunk can be too fragmentary to answer a question on its own (mid-explanation, missing the sentence that gives it meaning) | Retrieves a technically-relevant sentence fragment that reads as confusing or incomplete out of context |
| **500** | Moderate | Balanced | Balanced | Usually the most generally reliable default for prose-heavy documents like these |
| **1000** | Few, large | Lower -- a chunk can span multiple sub-topics | High -- almost always enough context to be self-contained | Similarity score gets "diluted": a chunk that's mostly about something else but happens to also mention the query topic can outrank a more precisely relevant (but smaller) chunk |

**General pattern to expect:** smaller chunks (200) tend to win on
*precision* -- the top result is narrowly on-topic -- but can lose on
*usability*, since a fragment plucked from the middle of a paragraph may
not stand alone. Larger chunks (1000) tend to win on *context* but can
lose on *precision*, especially on documents (like `doc_2_langchain` or
`doc_3_aws`, likely more heterogeneous than the research papers) where a
single large chunk drifts across several sub-topics.



## Cleanup — run this when you're done

**Don't skip this.** OpenSearch Serverless bills hourly for provisioned
compute whether or not you're actively querying it.

In [ ]:
# Delete the collection:
from src.vectorstore.opensearch_store import delete_collection
delete_collection()